# 03 – Klasická FFT (Linear Magnitude s DC, Welch PSD) a Sub-hertzová spektrometrie

### Fyzikální model a interpretace jevů:
1. **Ponechání stejnosměrné (DC) složky na 0 Hz (`fft_dc_magnitude`):**
   - Statické zatížení a posun základního optického výkonu $S_0$ se přímo projevují na frekvenci $0\text{ Hz}$. Nikdy ji neodstraňujeme před výpočtem průměru a celkové energie.
2. **Sub-hertzové pásmové filtry s krokem 0.2 Hz v pásmu 0.1–1.5 Hz (`fft_subband_*`):**
   - **Tepelná konvekce** vzduchu a médií kolem klubka optického vlákna vyvolává specifické nízkofrekvenční fluktuace (0.1 až 1.5 Hz). Zde sledujeme jemná pásma: `0.1–0.3`, `0.3–0.5`, `0.5–0.7`, `0.7–0.9`, `0.9–1.1`, `1.1–1.3`, `1.3–1.5` Hz a celkovou sub-hertzovou energii.
3. **STFT spektrogramy (Narrow vs. Wide):**
   - `STFT-Narrow` ($n_{\text{perseg}}=64$) pro sledování rezonancí s vysokým frekvenčním rozlišením.
   - `STFT-Wide` ($n_{\text{perseg}}=16$) pro časovou lokalizaci náběhových hran tepelných a mechanických přechodů.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from icaros.analysis.runner import load_windows_with_signals, run_single_silver_method
from icaros.db.s3 import read_parquet_from_s3

# 1. Načtení klouzavých oken a signálů (win128_stride32)
df_win, signals, fs = load_windows_with_signals('win128_stride32')
print(f'Načteno {len(df_win)} oken, vzorkovací frekvence fs = {fs} Hz')

# 2. Spuštění výpočtu metody fft_stft s novými sub-hertzovými pásmy
res = run_single_silver_method('fft_stft', df_win, signals, fs, 'win128_stride32')
print('Metriky separability (FFT + STFT):')
for k, v in res.items():
    if k != 's3_uri':
        print(f'  {k}: {v}')

# 3. Načtení výsledného Parquetu z MinIO / S3
df_fft = read_parquet_from_s3(res['s3_uri'])
print(f'Výsledný DataFrame: {df_fft.shape[0]} řádků, {df_fft.shape[1]} sloupců')


In [ ]:
# Přehled sub-hertzových energií a DC složky pro první okna
subband_cols = [
    'fft_dc_magnitude',
    'fft_subband_0_1_0_3_hz',
    'fft_subband_0_3_0_5_hz',
    'fft_subband_0_5_0_7_hz',
    'fft_subband_0_7_0_9_hz',
    'fft_subband_0_9_1_1_hz',
    'fft_subband_1_1_1_3_hz',
    'fft_subband_1_3_1_5_hz',
    'fft_subband_0_1_1_5_total',
]
disp_cols = ['measurement_code', 'inherited_label', 'inherited_target_value'] + subband_cols
display(df_fft[disp_cols].head(15))


In [ ]:
# Diagnostická a komparativní vizualizace FFT spektra, konvekce a STFT
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (A) STFT-Narrow spektrogram referenčního okna #0
stft_sample = np.stack(df_fft['stft_narrow_map'].iloc[0])
im0 = axes[0, 0].imshow(stft_sample, aspect='auto', origin='lower', cmap='viridis')
axes[0, 0].set_title(f'STFT-Narrow spektrogram okna #0 ({df_fft["inherited_label"].iloc[0]})')
axes[0, 0].set_xlabel('Časový rámec (okna v čase)')
axes[0, 0].set_ylabel('Frekvenční bin')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

# (B) FFT spektrum v sub-hertzovém pásmu 0–2.0 Hz podle tříd měření
freqs = np.array(df_fft['fft_frequencies'].iloc[0])
mask_sub = freqs <= 2.0
for lbl, grp in df_fft.groupby('inherited_label'):
    if len(grp) > 5 and not lbl.startswith('klid'):
        avg_spec = np.mean(np.vstack(grp['fft_spectrum'].values), axis=0)
        axes[0, 1].plot(freqs[mask_sub], avg_spec[mask_sub], label=str(lbl), alpha=0.85, linewidth=1.5)
axes[0, 1].axvspan(0.1, 1.5, color='orange', alpha=0.15, label='Konvekční pásmo 0.1–1.5 Hz')
axes[0, 1].set_title('Sub-hertzové FFT spektrum (Průměr dle teplotních tříd)')
axes[0, 1].set_xlabel('Frekvence [Hz]')
axes[0, 1].set_ylabel('Lineární magnituda |X(f)|')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)
axes[0, 1].legend(fontsize=7, loc='upper right')

# (C) Distribuce energií v jednotlivých pásmech 0.1–0.3 až 1.3–1.5 Hz
bands_labels = ['0.1-0.3', '0.3-0.5', '0.5-0.7', '0.7-0.9', '0.9-1.1', '1.1-1.3', '1.3-1.5']
band_feature_names = [f'fft_subband_{b.replace("-", "_").replace(".", "_")}_hz' for b in bands_labels]
avg_energies = [df_fft[col].mean() for col in band_feature_names]
axes[1, 0].bar(bands_labels, avg_energies, color='royalblue', edgecolor='navy', alpha=0.8)
axes[1, 0].set_title('Průměrná spektrální energie v sub-hertzových pásmech')
axes[1, 0].set_xlabel('Frekvenční sub-pásmo [Hz]')
axes[1, 0].set_ylabel('Energie pásma (Sum power)')
axes[1, 0].grid(True, axis='y', linestyle='--', alpha=0.5)

# (D) Separace: DC složka (0 Hz) vs. Celková sub-hertzová energie (0.1–1.5 Hz)
for lbl, grp in df_fft.groupby('inherited_label'):
    if len(grp) > 5:
        axes[1, 1].scatter(grp['fft_dc_magnitude'], grp['fft_subband_0_1_1_5_total'], s=14, alpha=0.6, label=str(lbl))
axes[1, 1].set_xlabel('FFT DC Magnitude (0 Hz složka)')
axes[1, 1].set_ylabel('Celková sub-hertzová energie (0.1–1.5 Hz)')
axes[1, 1].set_title('Separabilita: DC složka vs. Konvekční energie')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend(fontsize=7, loc='upper right')

plt.tight_layout()
plt.show()
